# Methods Justification and Biological Interpretation

This notebook explains the computational oncology pipeline step by step, focusing on why each method exists and what biological question it helps answer.

The goal is not only to run the code, but to understand how each computational step connects transcriptomic data to tumor subtype discovery, pathway biology, aging programs, and survival outcomes.

## Pipeline Overview

The project follows this analysis path:

```text
RNA-seq expression + clinical data
-> preprocessing
-> feature filtering and survival-based gene selection
-> scaling
-> PCA
-> GMM clustering
-> UMAP visualization
-> GSEA pathway enrichment
-> aging program scoring
-> survival analysis
```

Biologically, this workflow asks whether breast tumors contain molecularly distinct groups, whether those groups differ in biological pathway activity, and whether those differences relate to patient outcome.

## Methods Justification

### Why preprocessing?

RNA-seq matrices are high dimensional, noisy, and often contain missing or non-numeric values. Preprocessing creates a clean sample-by-gene matrix so downstream models compare tumors consistently.

### Why log transformation?

Gene expression values are typically right-skewed. Log2(expression + 1) reduces the influence of extremely high-expression genes and makes expression differences more comparable across genes.

### Why remove low-variance genes?

Genes with little variation across tumors are unlikely to explain subtype differences. Removing them reduces noise and computational burden while preserving genes that may reflect biological heterogeneity.

## Methods Justification: Feature Selection

### Why survival filtering changes results?

Survival-based feature selection biases clustering toward clinically relevant variation, improving the likelihood that discovered subtypes reflect prognostic heterogeneity.

Instead of clustering on all variable genes, the pipeline selects genes whose expression is individually associated with survival using Cox proportional hazards models.

Biological meaning: the resulting clusters are more likely to reflect tumor programs linked to aggressiveness, treatment resistance, immune escape, proliferation, or other processes that affect patient outcome.

Important caveat: this makes the clustering clinically informed rather than purely unsupervised. It can improve prognostic relevance, but it may also miss subtype structure unrelated to survival.

## Methods Justification: Scaling and PCA

### Why scale expression features?

Genes can have different expression ranges. Standard scaling gives each selected gene mean 0 and variance 1, preventing highly variable genes from dominating distance-based structure purely because of magnitude.

### Why PCA before clustering?

PCA is used as a denoising and dimensionality reduction step to improve clustering stability in high-dimensional gene expression space.

Biological meaning: PCA summarizes coordinated transcriptional programs. Instead of clustering tumors using hundreds or thousands of individual genes, the model clusters tumors using major axes of molecular variation.

## Methods Justification: Clustering

### Why GMM vs KMeans?

Gaussian Mixture Models were used to capture heterogeneous and overlapping tumor subpopulations, allowing probabilistic cluster assignments that better reflect biological variability.

KMeans assigns each tumor to the nearest centroid and assumes relatively spherical clusters. Tumors are often more complex: subtype boundaries can be gradual, mixed, or uncertain.

Biological meaning: GMMs better represent the idea that a tumor may share features with multiple molecular programs, rather than belonging perfectly to a single rigid group.

## Methods Justification: Choosing k

The clustering pipeline tests k values from 2 to 5 and selects the best value using silhouette score.

### Why silhouette score?

Silhouette score estimates whether samples are closer to their assigned cluster than to other clusters. Higher values suggest better separation.

Biological meaning: this helps choose a subtype number that balances within-subtype similarity and between-subtype distinction.

Important caveat: silhouette score is a mathematical criterion, not biological proof. The selected k should still be evaluated using pathway enrichment, survival differences, and domain knowledge.

## Methods Justification: UMAP

### Why UMAP improves structure?

UMAP was applied to capture nonlinear structure in transcriptomic data, improving visualization and interpretability of tumor subtypes.

The pipeline clusters in PCA space and uses UMAP for visualization. This is important because UMAP is powerful for display, but its 2D geometry should not be treated as the full biological structure.

Biological meaning: nearby tumors on the UMAP plot tend to have similar transcriptomic profiles, while separated groups suggest distinct molecular states.

## Methods Justification: Differential Expression

Differential expression compares one cluster against all other tumors.

### Why compare each cluster to the rest?

This identifies genes that characterize a subtype relative to the broader cohort.

Biological meaning: genes upregulated in a cluster can suggest active programs such as proliferation, immune infiltration, epithelial-mesenchymal transition, hypoxia, mitochondrial activity, or DNA damage response.

The pipeline ranks genes using both expression difference and statistical evidence, then uses that ranking for GSEA.

## Methods Justification: GSEA

### Why pathway enrichment instead of only marker genes?

Individual genes are noisy and difficult to interpret in isolation. GSEA tests whether predefined biological pathways are systematically enriched near the top or bottom of a ranked gene list.

Biological meaning: pathway-level results translate gene expression changes into interpretable tumor biology, such as immune activation, cell cycle activity, metabolic rewiring, WNT signaling, apoptosis, or inflammatory signaling.

The pipeline uses Hallmark, KEGG, Reactome, and GO Biological Process gene sets to capture complementary biological views.

## Methods Justification: NES and FDR

### Why use NES?

NES means normalized enrichment score. It allows pathway activity to be compared across pathways and clusters after accounting for gene set size.

Positive NES suggests a pathway is enriched among genes higher in the cluster. Negative NES suggests the pathway is relatively suppressed in that cluster.

### Why use FDR?

GSEA tests many pathways, so false discoveries are expected. FDR controls the expected proportion of false positives among significant pathways.

## Methods Justification: Aging Scores

The aging analysis groups enriched pathways into broad aging-related programs:

- inflammaging
- mitochondrial stress
- proteostasis loss
- cell cycle dysregulation
- identity drift
- DNA damage response

### Why score aging programs from pathways?

Aging is not usually represented by one gene or one pathway. It is a collection of coordinated biological processes. Aggregating GSEA results into aging programs provides a higher-level interpretation of how tumor subtypes may reflect age-associated biology.

## Methods Justification: Survival Analysis

### Why Kaplan-Meier curves?

Kaplan-Meier curves visualize survival probability over time for each inferred tumor subtype.

Biological meaning: if clusters have visibly different survival curves, the molecular subtypes may capture clinically meaningful tumor states.

### Why log-rank and Cox models?

The log-rank test evaluates whether survival curves differ between groups. Cox regression estimates the association between subtype membership and hazard while using time-to-event data.

Together, they test whether transcriptomic subtypes are only molecularly distinct or also clinically relevant.

## Code Map

| Pipeline stage | Main files | Biological meaning |
|---|---|---|
| Data loading | `src/data/loader/data_loader.py` | Reads RNA-seq and survival metadata |
| Expression preprocessing | `src/data/expression_preprocess.py` | Produces clean sample-by-gene tumor matrix |
| Clinical preprocessing | `src/data/clinical_preprocess.py` | Standardizes survival time and event labels |
| Alignment | `src/data/alignment.py` | Ensures expression and clinical rows refer to the same patients |
| Feature filtering | `src/features/feature_filtering.py` | Keeps genes with informative variation |
| Survival gene selection | `src/features/cox_feature_selection.py` | Prioritizes clinically associated genes |
| Scaling and PCA | `src/features/scaling.py`, `src/features/dimensionality_reduction.py` | Denoises high-dimensional expression structure |
| Clustering | `src/models/clustering.py`, `pipelines/clustering_pipeline.py` | Infers molecular tumor subtypes |
| GSEA | `pipelines/gsea_pipeline.py` | Converts gene rankings into pathway biology |
| Aging analysis | `analysis/aging_analysis.py` | Summarizes age-associated biological programs |
| Survival analysis | `src/evaluation/survival.py` | Tests clinical relevance of subtypes |
| Full orchestration | `scripts/run_full_pipeline.py` | Runs the complete workflow end to end |

In [ ]:
from pathlib import Path
import os

# Run notebook from the repository root, even when opened from notebooks/.
if Path.cwd().name == "notebooks":
    os.chdir("..")

print(Path.cwd())

## Inspect Preprocessed Data

The processed expression matrix should have tumors as rows and genes as columns. This structure is essential because every downstream model treats each row as one tumor sample.

In [ ]:
import pandas as pd

expr_path = Path("data/processed/expression_processed.csv")
clinical_path = Path("data/processed/clinical_processed.csv")

if expr_path.exists() and clinical_path.exists():
    expr = pd.read_csv(expr_path, index_col=0)
    clinical = pd.read_csv(clinical_path, index_col=0)
    print("Expression shape:", expr.shape)
    print("Clinical shape:", clinical.shape)
    display(expr.iloc[:5, :5])
    display(clinical.head())
else:
    print("Processed files not found. Run scripts/run_full_pipeline.py or run preprocessing first.")

## Inspect GSEA Results

GSEA results are the main bridge from machine learning clusters to biological interpretation. The most useful columns are:

- `Term`: pathway name
- `NES`: normalized enrichment score
- `FDR q-val`: multiple-testing adjusted significance
- `cluster`: tumor subtype
- `pathway_group`: broad biological category

In [ ]:
gsea_path = Path("results/gsea/all_gsea_results.csv")

if gsea_path.exists():
    gsea = pd.read_csv(gsea_path)
    print("GSEA shape:", gsea.shape)
    cols = [c for c in ["cluster", "gene_set", "Term", "NES", "FDR q-val", "pathway_group"] if c in gsea.columns]
    display(gsea[cols].head(10))
else:
    print("GSEA results not found. Run the full pipeline first.")

## Interpret Top Pathways Per Cluster

A subtype becomes biologically meaningful when it shows coherent pathway activity. For example:

- high cell cycle NES may indicate proliferative tumors
- high immune or interferon NES may suggest immune-infiltrated tumors
- high EMT, WNT, or TGF-beta NES may suggest invasive or stem-like programs
- high oxidative phosphorylation or mitochondrial terms may suggest metabolic differences

In [ ]:
if gsea_path.exists():
    gsea = pd.read_csv(gsea_path)
    if not gsea.empty:
        for cluster in sorted(gsea["cluster"].unique()):
            print(f"\nCluster {cluster}: top positive NES pathways")
            display(
                gsea[gsea["cluster"] == cluster]
                .sort_values("NES", ascending=False)
                [["Term", "NES", "FDR q-val", "pathway_group"]]
                .head(8)
            )
    else:
        print("GSEA file exists but contains no significant pathways.")

## Inspect Aging Scores

Aging scores summarize pathway enrichment into broader biological programs. This helps interpret whether particular tumor subtypes resemble age-associated states such as chronic inflammation, mitochondrial stress, or DNA damage response.

In [ ]:
aging_path = Path("results/aging/aging_scores.csv")

if aging_path.exists():
    aging = pd.read_csv(aging_path)
    display(aging)
else:
    print("Aging scores not found. Run the full pipeline first.")

## Figure Interpretation Guide

### UMAP plot

Shows whether tumors form visually separated molecular groups. Separation suggests transcriptomic heterogeneity.

### Pathway heatmap

Shows which pathways are activated or suppressed in each subtype. This is usually the most biologically interpretable figure.

### Aging heatmap

Shows whether subtypes differ in aging-related biological programs.

### Kaplan-Meier plot

Shows whether the molecular groups have different survival trajectories.

In [1]:
from IPython.display import Image, display

figure_paths = [
    Path("results/figures/umap.png"),
    Path("results/figures/pathway_heatmap.png"),
    Path("results/figures/aging_heatmap.png"),
    Path("results/figures/km_plot.png"),
]

for path in figure_paths:
    print(path)
    if path.exists():
        display(Image(filename=str(path)))
    else:
        print("Missing. Run the full pipeline to generate this figure.")

NameError: name 'Path' is not defined

## Limitations and Interpretation Caveats

- Survival-based feature selection can increase clinical relevance but may bias subtype discovery toward prognostic signals.
- PCA removes noise but can also remove weaker biological signals.
- UMAP is useful for visualization, but distances in 2D should not be overinterpreted.
- GSEA depends on gene set databases and pathway annotations, which are incomplete and sometimes overlapping.
- Pathway keyword grouping is a useful summary, but it is not a substitute for expert biological curation.
- Survival associations are exploratory unless validated in an independent cohort.

## Summary

This pipeline uses transcriptomic data to identify candidate tumor subtypes, then asks whether those subtypes are biologically interpretable and clinically meaningful.

The core reasoning is:

1. Clean and align molecular and clinical data.
2. Reduce noise while preserving clinically relevant gene expression variation.
3. Cluster tumors into candidate molecular subtypes.
4. Interpret subtypes using pathway enrichment and aging biology.
5. Evaluate whether subtypes correspond to survival differences.

A strong result is not just a visually separated cluster. It is a cluster that has coherent pathway biology and a plausible relationship to clinical outcome.